In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv(
    "../data/microsoft_file_metadata.txt",
    sep="\t",
    header=None
)

print(df.shape)
df.head()

(3350, 13)


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,1,000e46ce402500409354,39c4a71ba8949d8e0a2d,39c4a71ba8949d8e0a2d,604879208,c,NTFS,2932253696,4310013952,126142230903260000,512,459007,502
1,7,0ae9d8ff4a2fde36885c,3455bd0b4c1501c7d899,06fcb9effa6b460f60e9,773854956,c,FAT32,5109891072,8430841856,126137892909020000,4096,6,502
2,12,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,618801528,f,NTFS,7059025920,18194284544,126134307922350000,4096,459007,198
3,13,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,-1878414956,c,NTFS,1083307520,2146765312,126134307738280000,512,459007,198
4,14,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,-1604198837,d,NTFS,5591822336,6950326272,126134307850470000,4096,459007,198


In [3]:
df.columns = [
    "SnapshotID",
    "FileID",
    "ParentDirectoryID",
    "FileNameHash",
    "FileExtension",
    "FileType",
    "Unknown",
    "FileSize",
    "AllocatedSize",
    "Timestamp",
    "ClusterSize",
    "Attributes",
    "Meta"
]

df.head()

,SnapshotID,FileID,ParentDirectoryID,FileNameHash,FileExtension,FileType,Unknown,FileSize,AllocatedSize,Timestamp,ClusterSize,Attributes,Meta
0,1,000e46ce402500409354,39c4a71ba8949d8e0a2d,39c4a71ba8949d8e0a2d,604879208,c,NTFS,2932253696,4310013952,126142230903260000,512,459007,502
1,7,0ae9d8ff4a2fde36885c,3455bd0b4c1501c7d899,06fcb9effa6b460f60e9,773854956,c,FAT32,5109891072,8430841856,126137892909020000,4096,6,502
2,12,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,618801528,f,NTFS,7059025920,18194284544,126134307922350000,4096,459007,198
3,13,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,-1878414956,c,NTFS,1083307520,2146765312,126134307738280000,512,459007,198
4,14,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,-1604198837,d,NTFS,5591822336,6950326272,126134307850470000,4096,459007,198


In [4]:
numeric_cols = [
    "FileSize",
    "AllocatedSize",
    "Timestamp",
    "ClusterSize",
    "Attributes"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3350 entries, 0 to 3349
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   SnapshotID         3350 non-null   int64 
 1   FileID             3350 non-null   object
 2   ParentDirectoryID  3350 non-null   object
 3   FileNameHash       3350 non-null   object
 4   FileExtension      3350 non-null   int64 
 5   FileType           3350 non-null   object
 6   Unknown            3350 non-null   object
 7   FileSize           3350 non-null   int64 
 8   AllocatedSize      3350 non-null   int64 
 9   Timestamp          3350 non-null   int64 
 10  ClusterSize        3350 non-null   int64 
 11  Attributes         3350 non-null   int64 
 12  Meta               3350 non-null   int64 
dtypes: int64(8), object(5)
memory usage: 340.4+ KB


In [5]:
df["FileSizeMB"] = df["FileSize"] / (1024 * 1024)

df["AllocatedSizeMB"] = (
    df["AllocatedSize"] / (1024 * 1024)
)

df["FileDate"] = pd.to_datetime(
    (df["Timestamp"] - 116444736000000000) / 10000000,
    unit="s"
)

latest_date = df["FileDate"].max()

df["FileAgeYears"] = (
    (latest_date - df["FileDate"]).dt.days
    / 365
)

df.head()

,SnapshotID,FileID,ParentDirectoryID,FileNameHash,FileExtension,FileType,Unknown,FileSize,AllocatedSize,Timestamp,ClusterSize,Attributes,Meta,FileSizeMB,AllocatedSizeMB,FileDate,FileAgeYears
0,1,000e46ce402500409354,39c4a71ba8949d8e0a2d,39c4a71ba8949d8e0a2d,604879208,c,NTFS,2932253696,4310013952,126142230903260000,512,459007,502,2796.415039,4110.349609,2000-09-23 22:51:30.325999975,4.482192
1,7,0ae9d8ff4a2fde36885c,3455bd0b4c1501c7d899,06fcb9effa6b460f60e9,773854956,c,FAT32,5109891072,8430841856,126137892909020000,4096,6,502,4873.171875,8040.277344,2000-09-18 22:21:30.901999950,4.495890
2,12,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,618801528,f,NTFS,7059025920,18194284544,126134307922350000,4096,459007,198,6732.011719,17351.421875,2000-09-14 18:46:32.235000014,4.506849
3,13,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,-1878414956,c,NTFS,1083307520,2146765312,126134307738280000,512,459007,198,1033.122559,2047.314941,2000-09-14 18:46:13.827999949,4.506849
4,14,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,-1604198837,d,NTFS,5591822336,6950326272,126134307850470000,4096,459007,198,5332.777344,6628.347656,2000-09-14 18:46:25.047000051,4.506849


In [6]:
df["FileSizeMB"] = df["FileSize"] / (1024 * 1024)

df["AllocatedSizeMB"] = (
    df["AllocatedSize"] / (1024 * 1024)
)

df["FileDate"] = pd.to_datetime(
    (df["Timestamp"] - 116444736000000000) / 10000000,
    unit="s"
)

latest_date = df["FileDate"].max()

df["FileAgeYears"] = (
    (latest_date - df["FileDate"]).dt.days
    / 365
)

df.head()

,SnapshotID,FileID,ParentDirectoryID,FileNameHash,FileExtension,FileType,Unknown,FileSize,AllocatedSize,Timestamp,ClusterSize,Attributes,Meta,FileSizeMB,AllocatedSizeMB,FileDate,FileAgeYears
0,1,000e46ce402500409354,39c4a71ba8949d8e0a2d,39c4a71ba8949d8e0a2d,604879208,c,NTFS,2932253696,4310013952,126142230903260000,512,459007,502,2796.415039,4110.349609,2000-09-23 22:51:30.325999975,4.482192
1,7,0ae9d8ff4a2fde36885c,3455bd0b4c1501c7d899,06fcb9effa6b460f60e9,773854956,c,FAT32,5109891072,8430841856,126137892909020000,4096,6,502,4873.171875,8040.277344,2000-09-18 22:21:30.901999950,4.495890
2,12,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,618801528,f,NTFS,7059025920,18194284544,126134307922350000,4096,459007,198,6732.011719,17351.421875,2000-09-14 18:46:32.235000014,4.506849
3,13,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,-1878414956,c,NTFS,1083307520,2146765312,126134307738280000,512,459007,198,1033.122559,2047.314941,2000-09-14 18:46:13.827999949,4.506849
4,14,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,-1604198837,d,NTFS,5591822336,6950326272,126134307850470000,4096,459007,198,5332.777344,6628.347656,2000-09-14 18:46:25.047000051,4.506849


In [7]:
print("Rows:", len(df))

print("\nFile Size Stats:")
print(df["FileSizeMB"].describe())

print("\nAge Stats:")
print(df["FileAgeYears"].describe())

Rows: 3350

File Size Stats:
count      3350.000000
mean      14606.145507
std       22850.308517
min           0.007812
25%        2058.157227
50%        6767.734375
75%       18095.619141
max      424811.976562
Name: FileSizeMB, dtype: float64

Age Stats:
count    3350.000000
mean        2.361043
std         1.373310
min         0.000000
25%         1.400000
50%         2.441096
75%         3.430137
max         4.515068
Name: FileAgeYears, dtype: float64


In [8]:
features = [
    "FileSizeMB",
    "AllocatedSizeMB",
    "FileAgeYears",
    "ClusterSize",
    "Attributes"
]

X = df[features]

print(X.shape)
X.head()

(3350, 5)


,FileSizeMB,AllocatedSizeMB,FileAgeYears,ClusterSize,Attributes
0,2796.415039,4110.349609,4.482192,512,459007
1,4873.171875,8040.277344,4.495890,4096,6
2,6732.011719,17351.421875,4.506849,4096,459007
3,1033.122559,2047.314941,4.506849,512,459007
4,5332.777344,6628.347656,4.506849,4096,459007


In [9]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

(3350, 5)


In [10]:
iso_model = IsolationForest(
    n_estimators=300,
    contamination=0.05,
    random_state=42
)

iso_model.fit(X_scaled)

print("Isolation Forest Trained")

Isolation Forest Trained


In [11]:
df["Anomaly"] = iso_model.predict(
    X_scaled
)

df["AnomalyScore"] = (
    iso_model.decision_function(X_scaled)
)

df.head()

,SnapshotID,FileID,ParentDirectoryID,FileNameHash,FileExtension,FileType,Unknown,FileSize,AllocatedSize,Timestamp,ClusterSize,Attributes,Meta,FileSizeMB,AllocatedSizeMB,FileDate,FileAgeYears,Anomaly,AnomalyScore
0,1,000e46ce402500409354,39c4a71ba8949d8e0a2d,39c4a71ba8949d8e0a2d,604879208,c,NTFS,2932253696,4310013952,126142230903260000,512,459007,502,2796.415039,4110.349609,2000-09-23 22:51:30.325999975,4.482192,1,0.110294
1,7,0ae9d8ff4a2fde36885c,3455bd0b4c1501c7d899,06fcb9effa6b460f60e9,773854956,c,FAT32,5109891072,8430841856,126137892909020000,4096,6,502,4873.171875,8040.277344,2000-09-18 22:21:30.901999950,4.495890,1,0.112721
2,12,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,618801528,f,NTFS,7059025920,18194284544,126134307922350000,4096,459007,198,6732.011719,17351.421875,2000-09-14 18:46:32.235000014,4.506849,1,0.144120
3,13,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,-1878414956,c,NTFS,1083307520,2146765312,126134307738280000,512,459007,198,1033.122559,2047.314941,2000-09-14 18:46:13.827999949,4.506849,1,0.111919
4,14,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,-1604198837,d,NTFS,5591822336,6950326272,126134307850470000,4096,459007,198,5332.777344,6628.347656,2000-09-14 18:46:25.047000051,4.506849,1,0.155648


In [12]:
print(
    df["Anomaly"].value_counts()
)

Anomaly
 1    3182
-1     168
Name: count, dtype: int64


In [13]:
anomalies = df[df["Anomaly"] == -1]

anomalies = anomalies.sort_values(
    by="AnomalyScore"
)

print("Total anomalies:", len(anomalies))

anomalies[
    [
        "FileSizeMB",
        "AllocatedSizeMB",
        "FileAgeYears",
        "AnomalyScore"
    ]
].head(20)

Total anomalies: 168


,FileSizeMB,AllocatedSizeMB,FileAgeYears,AnomalyScore
2600,424811.976562,903028.679688,1.427397,-0.160314
2197,130720.093750,156281.750000,1.397260,-0.159780
2599,347170.023438,903028.679688,1.427397,-0.157408
2601,297849.984375,416772.191406,1.427397,-0.141768
2572,217255.605469,476239.390625,1.400000,-0.141334
2576,201774.761719,476247.609375,1.400000,-0.140666
2682,92281.687500,114442.968750,1.402740,-0.129168
2771,191374.187500,238409.902344,0.438356,-0.125973
2282,87102.000000,114442.968750,1.424658,-0.125223
3061,78125.187500,107204.468750,0.424658,-0.125002


In [14]:
print("NORMAL FILES")
print(
    df[df["Anomaly"] == 1][
        [
            "FileSizeMB",
            "AllocatedSizeMB",
            "FileAgeYears"
        ]
    ].describe()
)

print("\nANOMALY FILES")
print(
    df[df["Anomaly"] == -1][
        [
            "FileSizeMB",
            "AllocatedSizeMB",
            "FileAgeYears"
        ]
    ].describe()
)

NORMAL FILES
         FileSizeMB  AllocatedSizeMB  FileAgeYears
count   3182.000000      3182.000000   3182.000000
mean   12756.380454     23287.524957      2.369322
std    15802.348651     24682.622858      1.354669
min        0.007812         7.785156      0.284932
25%     2062.292236      6695.048828      1.400000
50%     6603.640625     15249.566406      2.441096
75%    17186.977539     37666.046875      3.429452
max    92308.820312    208386.777344      4.515068

ANOMALY FILES
          FileSizeMB  AllocatedSizeMB  FileAgeYears
count     168.000000       168.000000    168.000000
mean    49641.576454     81467.917620      2.204224
std     66440.285209    125537.781090      1.685184
min        94.375000      1552.906250      0.000000
25%      2044.843750      3749.953125      0.437671
50%     19780.712891     31988.812500      1.427397
75%     92059.620117    114470.937500      4.487671
max    424811.976562    903028.679688      4.509589


In [15]:
df["DigitalWasteScore"] = (
    100 *
    (
        (
            df["AnomalyScore"]
            - df["AnomalyScore"].min()
        )
        /
        (
            df["AnomalyScore"].max()
            - df["AnomalyScore"].min()
        )
    )
)

df["DigitalWasteScore"] = (
    100 - df["DigitalWasteScore"]
)

df["DigitalWasteScore"].describe()

count    3350.000000
mean       20.529546
std        17.510696
min         0.000000
25%         5.836037
50%        17.395405
75%        29.324218
max       100.000000
Name: DigitalWasteScore, dtype: float64

In [16]:
conditions = [
    df["DigitalWasteScore"] >= 80,
    df["DigitalWasteScore"] >= 60,
    df["DigitalWasteScore"] >= 40
]

choices = [
    "Critical",
    "High",
    "Medium"
]

df["WasteRisk"] = np.select(
    conditions,
    choices,
    default="Low"
)

print(df["WasteRisk"].value_counts())

WasteRisk
Low         2868
Medium       349
High         112
Critical      21
Name: count, dtype: int64


In [17]:
def generate_recommendation(row):

    if row["WasteRisk"] == "Critical":
        return "Delete or archive immediately"

    elif row["WasteRisk"] == "High":
        return "Move to cold storage"

    elif row["WasteRisk"] == "Medium":
        return "Review file usage"

    else:
        return "Keep active"

df["Recommendation"] = (
    df.apply(
        generate_recommendation,
        axis=1
    )
)

print(
    df[
        [
            "DigitalWasteScore",
            "WasteRisk",
            "Recommendation"
        ]
    ].head()
)

   DigitalWasteScore WasteRisk Recommendation
0          29.727076       Low    Keep active
1          29.096797       Low    Keep active
2          20.942953       Low    Keep active
3          29.305119       Low    Keep active
4          17.949273       Low    Keep active


In [18]:
output_cols = [
    "FileID",
    "FileSizeMB",
    "FileAgeYears",
    "AnomalyScore",
    "DigitalWasteScore",
    "WasteRisk",
    "Recommendation"
]

df[output_cols].to_csv(
    "../data/final_analysis_results.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully


In [20]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(
    iso_model,
    "../models/eco_byte_anomaly_model.pkl"
)

joblib.dump(
    scaler,
    "../models/eco_byte_scaler.pkl"
)

print("Model Saved Successfully")

Model Saved Successfully
